# AI-Powered Chatbot
## IntelliChat – AI-Powered Knowledge Assistant

## 1. Problem statement

Users need to locate precise information in long documents and ask contextual follow-up questions. Manual searching is slow, simple keyword search loses conversational intent, and a general LLM does not automatically know private files. Unsupported generated answers can appear authoritative.

**Proposed solution:** a general conversational assistant with an additional Retrieval-Augmented Generation (RAG) mode. It retrieves document passages, supplies them to the model, and exposes checked source quotations.

**Objectives:** support general chat; ingest PDF/TXT/DOCX; preserve provenance; retrieve relevant chunks; resolve follow-ups; return a safe insufficient-evidence response; expose REST APIs; separate UI and services; test failures as well as successful paths.

**Scope:** local capstone collections, single-process in-memory state, configurable OpenAI/Ollama adapters. No OCR, authentication, or persistent database is claimed.

## 2. Architecture and execution flow

```text
User → Streamlit → HTTP → FastAPI → Services → LangChain → LLM
                                           └→ Retriever → FAISS

Upload → validate → extract text → overlapping chunks → embeddings → index
Question + history → standalone query → retrieval → context + prompt
                  → model → validate source IDs and quotes → answer + sources
```

The frontend contains presentation and HTTP calls. Services own business logic. General and document conversation histories are separate. Retrieval uses normalized vectors: their inner product equals cosine similarity. Document vectors are reused across questions.

| Layer | Technology |
|---|---|
| UI / HTTP | Streamlit / HTTPX |
| Backend / validation | FastAPI / Pydantic |
| Orchestration | LangChain Core prompts, Runnables, messages and parsers |
| Extraction | pypdf / python-docx |
| Retrieval | FAISS / NumPy |
| Providers | langchain-openai / langchain-ollama |
| Testing | pytest / FastAPI TestClient / Streamlit AppTest |

## 3. Environment setup

Select a **Python 3.12** kernel. In a fresh environment, install dependencies before running the remaining cells. Installation is not automatically performed by Run All.

In [15]:
import json
import os
import sys
import tempfile
from pathlib import Path
from datetime import datetime, timezone

from IPython.display import display, Markdown, Code

started_at = datetime.now(timezone.utc)
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "backend/main.py").is_file()
             and (p / "sample_data/company_policy.txt").is_file()), None)
if ROOT is None:
    notebook_path = Path.cwd() / "project.ipynb"
    if not notebook_path.is_file():
        raise FileNotFoundError("Place project.ipynb in the kernel working directory, or open it from the project root.")
    bundle = json.loads(notebook_path.read_text(encoding="utf-8"))["metadata"]["intellichat_source"]
    ROOT = Path(tempfile.mkdtemp(prefix="intellichat-notebook-"))
    for relative, contents in bundle.items():
        target = (ROOT / relative).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe path in embedded source bundle")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(contents, encoding="utf-8")
    print("Extracted embedded implementation into a new temporary project.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.version.split()[0])
print("Project:", ROOT)


Python: 3.12.10
Project: c:\VS Projects\Assignment\AI-Powered Chatbot


## 4. Configuration and API contracts

Settings centrally validate chunk sizes, limits, providers, timeouts, and CORS. The offline experiment explicitly overrides provider keys and avoids .env. Production secrets are read by the backend only. Pydantic validates messages, session identifiers, mode, Top-K, and temperature before a model call.

The source appendix contains the complete Settings and schema implementations.

In [16]:
from backend.core.config import Settings
from backend.schemas.chat import ChatRequest
from pydantic import ValidationError

settings = Settings(
    _env_file=None, llm_provider="openai", embedding_provider="openai",
    llm_api_key="", embedding_api_key="", chunk_size=1000, chunk_overlap=150,
    top_k=4, min_similarity=0.2, log_level="ERROR",
)
print({name: getattr(settings, name) for name in ("chunk_size", "chunk_overlap", "top_k", "min_similarity")})
valid_request = ChatRequest(session_id="notebook", message="What is the remote work policy?", mode="rag")
print(valid_request.model_dump())
try:
    ChatRequest(session_id="notebook", message="   ")
except ValidationError:
    print("PASS: whitespace-only messages are rejected.")

{'chunk_size': 1000, 'chunk_overlap': 150, 'top_k': 4, 'min_similarity': 0.2}
{'session_id': 'notebook', 'message': 'What is the remote work policy?', 'mode': 'rag', 'top_k': None, 'temperature': 0.2}
PASS: whitespace-only messages are rejected.


## 5. General Chat — Conversational AI interaction

This section explicitly implements the **General Chat Conversational AI** scope of IntelliChat.

Unlike RAG mode, General Chat does **not** retrieve document chunks. The user message is combined with bounded conversation history and sent to the configured LLM through the LangChain `GENERAL_PROMPT`. This enables natural multi-turn interaction such as:

1. User asks a question.
2. Assistant answers using the configured LLM.
3. User asks a follow-up using words such as “it”, “that”, or “what are its benefits?”.
4. Previous turns are supplied to the LLM so it can resolve the conversational context.
5. Only a successful answer is appended to memory.

The notebook uses an injected offline model for reproducible execution. A separate optional cell below shows how the **same chain can use OpenAI or Ollama** without hard-coding credentials.

**Important distinction:** General Chat provides conversational AI, while Document/RAG Chat provides answers grounded in uploaded PDF/TXT/DOCX content with source quotations. The two histories are intentionally kept separate.


In [17]:
# General Chat: build the same LangChain prompt used by the application.
#
# GENERAL_PROMPT contains:
#   - a system instruction defining IntelliChat as a helpful assistant,
#   - a MessagesPlaceholder for previous conversation turns,
#   - the current user question.
#
# Keeping the prompt in backend.ai.prompts means the notebook and the actual
# FastAPI application use the same conversational behavior rather than having
# two different implementations.

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from backend.ai.prompts import GENERAL_PROMPT

general_history = []

def run_general_chat_turn(question: str, history: list, llm) -> tuple[str, list]:
    """Run one conversational turn and return the answer plus updated history.

    Parameters
    ----------
    question:
        The latest user message.
    history:
        A list containing alternating HumanMessage / AIMessage objects.
    llm:
        A LangChain-compatible chat model or the deterministic notebook
        substitute used for offline verification.

    Design choices
    --------------
    * History is passed to the prompt so follow-up questions retain context.
    * The answer is converted to a plain string using StrOutputParser.
    * Memory is updated only after a successful model response.
    * The history is bounded to the configured number of turns to avoid
      unbounded prompt growth.
    """
    question = question.strip()
    if not question:
        raise ValueError("Question must not be empty.")

    # Construct the conversational chain:
    #   GENERAL_PROMPT -> LLM -> plain text answer
    chain = GENERAL_PROMPT | llm | StrOutputParser()

    # Invoke the model with the current history and latest user question.
    answer = chain.invoke({
        "history": history,
        "question": question,
    }).strip()

    if not answer:
        raise ValueError("The model returned an empty response.")

    # Commit the conversation pair only after the LLM call succeeds.
    updated_history = [
        *history,
        HumanMessage(content=question),
        AIMessage(content=answer),
    ]

    # Keep only the configured number of most recent turns.
    max_messages = 2 * settings.max_history_turns
    updated_history = updated_history[-max_messages:]

    return answer, updated_history


print("General Chat chain ready.")
print("Prompt inputs:", GENERAL_PROMPT.input_variables)


General Chat chain ready.
Prompt inputs: ['history', 'question']


### 5.1 Offline conversational demonstration

The next cell uses the notebook's deterministic model substitute. This keeps the notebook reproducible and does not require an API key, while still exercising the **real LangChain prompt + history + chain code** used for General Chat.


In [18]:
# A small deterministic model is used here so the notebook can be executed
# without network access or an LLM API key. It deliberately echoes the
# conversational context instead of pretending to measure LLM quality.

from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

def offline_general_chat_model(prompt_value):
    """Deterministic stand-in for a chat LLM used only for notebook testing."""
    messages = prompt_value.to_messages()
    user_question = messages[-1].content

    # Count prior turns to prove that conversation history is reaching the model.
    prior_turns = sum(isinstance(m, HumanMessage) for m in messages[:-1])

    return AIMessage(
        content=(
            f"Offline conversational response to: {user_question} "
            f"(conversation context contains {prior_turns} previous user turn(s))."
        )
    )

offline_general_llm = RunnableLambda(offline_general_chat_model)

# Multi-turn conversation: the second and third questions are intentionally
# dependent on the previous discussion.
general_history = []

answer_1, general_history = run_general_chat_turn(
    "What is Retrieval-Augmented Generation?", general_history, offline_general_llm
)
answer_2, general_history = run_general_chat_turn(
    "What are its main benefits?", general_history, offline_general_llm
)
answer_3, general_history = run_general_chat_turn(
    "How is that different from a normal LLM chat?", general_history, offline_general_llm
)

print("TURN 1:", answer_1)
print("TURN 2:", answer_2)
print("TURN 3:", answer_3)
print("\nStored messages:", len(general_history))

# Verification: three user/assistant pairs should be stored.
assert len(general_history) == 6
assert isinstance(general_history[0], HumanMessage)
assert isinstance(general_history[1], AIMessage)
assert isinstance(general_history[-2], HumanMessage)
assert isinstance(general_history[-1], AIMessage)

print("PASS: General Chat supports multi-turn conversational memory.")


TURN 1: Offline conversational response to: What is Retrieval-Augmented Generation? (conversation context contains 0 previous user turn(s)).
TURN 2: Offline conversational response to: What are its main benefits? (conversation context contains 1 previous user turn(s)).
TURN 3: Offline conversational response to: How is that different from a normal LLM chat? (conversation context contains 2 previous user turn(s)).

Stored messages: 6
PASS: General Chat supports multi-turn conversational memory.


In [19]:
# ============================================================
# Load application settings AFTER .env has been loaded
# ============================================================

from backend.core.config import Settings

settings = Settings()

print("Application settings:")
print("LLM Provider :", settings.llm_provider)
print("LLM Model    :", settings.llm_model)
print(
    "LLM API Key  :",
    "Configured" if settings.llm_api_key else "MISSING"
)

Application settings:
LLM Provider : openai
LLM Model    : gpt-4.1-mini
LLM API Key  : Configured


### 5.2 Optional live OpenAI / Ollama General Chat

For the final application, `Settings` selects either OpenAI or Ollama. Credentials are read from the environment by the backend; no secret is stored in this notebook.

Set the required environment variables in `.env`, then change `RUN_LIVE_GENERAL_CHAT` to `True`. The same `run_general_chat_turn()` function is used, so the notebook demonstrates the real provider path without duplicating application logic.


In [20]:
# LIVE DEMONSTRATION
# Keep this False for a reproducible submission without API credentials.
RUN_LIVE_GENERAL_CHAT = True

if RUN_LIVE_GENERAL_CHAT:
    from backend.ai.llm import create_llm

    # create_llm reads the provider/model configuration from Settings.
    # Supported providers in this project are OpenAI and Ollama.
    live_llm = create_llm(settings, temperature=0.2)

    live_history = []
    live_answer, live_history = run_general_chat_turn(
        "Explain what this IntelliChat application does.",
        live_history,
        live_llm,
    )

    print("LIVE GENERAL CHAT RESPONSE:")
    print(live_answer)
else:
    print(
        "Live provider execution is disabled. "
        "Set RUN_LIVE_GENERAL_CHAT=True after configuring OpenAI/Ollama credentials."
    )


LIVE GENERAL CHAT RESPONSE:
The IntelliChat application is an AI-powered chat assistant designed to provide clear, accurate, and concise responses to user queries. It helps users by understanding their questions and delivering helpful information or guidance in a professional manner. If it encounters uncertainty about a topic, it acknowledges that rather than providing potentially incorrect information. Essentially, IntelliChat serves as an intelligent conversational partner to assist with a wide range of topics.


## 6. Document ingestion and provenance

The sample is the fictional NovaTech Solutions employee handbook. The loader sanitizes names, rejects unsupported/corrupt/empty files, bounds extracted text and DOCX expansion, and never executes uploaded content. PDF pages are one-based; TXT and DOCX have no reliable pagination, so their page value stays null.

In [21]:
from backend.ai.document_loader import load_document

sample_path = ROOT / "sample_data/company_policy.txt"
sample_bytes = sample_path.read_bytes()
pages = load_document(sample_path.name, sample_bytes, "notebook-policy", settings.max_extracted_chars)
print("Extracted sections/pages:", len(pages))
print("Characters:", sum(len(page.page_content) for page in pages))
print("Metadata:", pages[0].metadata)
print("\nPreview:\n", pages[0].page_content[:600])
assert pages[0].metadata["page"] is None

Extracted sections/pages: 1
Characters: 6074
Metadata: {'file_name': 'company_policy.txt', 'source': 'company_policy.txt', 'page': None, 'document_id': 'notebook-policy'}

Preview:
 NovaTech Solutions
Employee Handbook — Fictional Demonstration Dataset
Effective date: 1 January 2026
This company and all policies are fictional and provided only for IntelliChat demonstrations.

1. Working hours
The standard working week is Monday to Friday, 40 hours per week. Normal office hours are
9:00 a.m. to 6:00 p.m., including a one-hour unpaid lunch break. Teams observe core collaboration
hours from 10:00 a.m. to 4:00 p.m. Employees may agree flexible start and finish times with their
manager, provided service coverage and weekly hours are maintained. Overtime requires written
manage


## 7. Overlapping text chunks

RecursiveCharacterTextSplitter prefers natural text boundaries while respecting a character limit. Overlap helps retain meaning spanning two chunks. Metadata is copied to every chunk; a stable chunk ID links retrieval to evidence.

In [22]:
from backend.ai.text_splitter import split_documents

chunks = split_documents(pages, settings)
print("Chunk count:", len(chunks))
print("Lengths:", [len(chunk.page_content) for chunk in chunks])
assert all(len(chunk.page_content) <= settings.chunk_size for chunk in chunks)
assert len({chunk.metadata["chunk_id"] for chunk in chunks}) == len(chunks)
display(Code(chunks[1].page_content, language="text"))
print(chunks[1].metadata)

Chunk count: 9
Lengths: [645, 928, 808, 533, 648, 585, 535, 550, 826]


2. Annual leave
Full-time employees receive 24 days of paid annual leave per calendar year, accrued at two days
per completed month. New starters receive a prorated allowance. Submit annual leave requests
through the HR portal at least seven calendar days before the intended absence. The manager
approves requests based on staffing requirements. Up to five unused days may be carried into
the next year and must be used by 31 March. Public holidays do not reduce annual leave balances.

3. Sick leave
Employees receive 10 days of paid sick leave per calendar year. Notify the line manager before
10:00 a.m. on the first day of sickness, or as soon as reasonably possible in an emergency.
A medical certificate is required after more than two consecutive working days of absence.
Sick leave does not carry forward to the next year. Health records are handled confidentially
by HR and are accessible only to authorized personnel.

{'file_name': 'company_policy.txt', 'source': 'company_policy.txt', 'page': None, 'document_id': 'notebook-policy', 'start_index': 647, 'chunk_id': 'notebook-policy_1'}


## 8. Embedding adapter and real FAISS search

For a reproducible no-cost demonstration, the following **test-only** adapter maps topic words into small vectors. These are not semantic embeddings and cannot demonstrate production retrieval quality. The application normally uses the configurable embedding factories shown in the source appendix.

The FAISS implementation is real: it normalizes float32 vectors, uses IndexFlatIP, stages updates atomically, and preserves the corresponding LangChain Documents.

In [23]:
from tests.conftest import KeywordEmbeddings
from backend.ai.vector_store import VectorStore

# Expose the full offline embedding implementation for academic inspection.
import inspect
display(Code(inspect.getsource(KeywordEmbeddings), language="python"))

embeddings = KeywordEmbeddings()
vector_store = VectorStore(embeddings)
vector_store.add_documents(chunks)
hits = vector_store.search("What is the remote work policy?", 4, settings.min_similarity)
for document, score in hits:
    print(document.metadata["chunk_id"], "cosine =", round(score, 4))
    print(document.page_content[:160].replace("\n", " "), "\n")
assert hits
assert embeddings.document_calls == 1
assert embeddings.query_calls == 1

class KeywordEmbeddings(Embeddings):
    """Map topic words to small vectors to test real FAISS operations offline."""

    vocabulary = ("leave", "remote", "expense", "password", "training", "ceo", "join", "salary")

    def __init__(self):
        """Track document/query embedding counts to detect redundant embedding work."""
        self.document_calls = 0
        self.query_calls = 0

    def vector(self, text: str) -> list[float]:
        """Count fixed topic stems and add a small nonzero component."""
        words = re.findall(r"\w+", text.lower())
        return [
            float(sum(word.startswith(topic) for word in words)) for topic in self.vocabulary
        ] + [0.01]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Return deterministic document vectors and record the batch call."""
        self.document_calls += 1
        return [self.vector(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        """Return the same vector space for the search query."""
        self.query_calls += 1
        return self.vector(text)

2026-09-26 19:25:35,979 INFO backend.ai.vector_store Embedding generation started: chunks=9
2026-09-26 19:25:35,985 INFO backend.ai.vector_store Embedding generation completed; vector store updated: total=9


notebook-policy_2 cosine = 1.0
4. Remote and hybrid work Eligible employees may work remotely up to two days per week with manager approval. The default hybrid schedule includes at least thre 



## 9. Prompts, grounded answers, and the offline model substitute

Three prompts handle general dialogue, standalone-query rewriting, and RAG. The RAG model must return JSON with an answer and evidence quotations. The service validates every source ID and quote against actual retrieved chunks.

The Runnable below is explicitly **a rule-based test double, not an LLM**. It extracts matching sentences from retrieved context for known demonstration topics. It never reads expected answers from the evaluation dataset. Its high scores, if any, are not evidence of language-model accuracy. A real provider is enabled only in the optional live section.

In [24]:
import re
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from backend.ai.prompts import FALLBACK

model_calls = []

def offline_model(prompt):
    """Return controlled outputs to exercise the real orchestration without a model API."""
    messages = prompt.to_messages()
    system = messages[0].content
    latest = messages[-1].content
    if system.startswith("Rewrite"):
        query = ("How much unused annual leave can be carried forward?"
                 if "carried" in latest.lower() else latest)
        model_calls.append({"stage": "rewrite", "query": query})
        return AIMessage(content=query)
    if "Retrieved context (JSON data" not in latest:
        model_calls.append({"stage": "general", "history_messages": len(messages) - 2})
        return AIMessage(content="Offline demonstration response: RAG retrieves document context before generating an answer.")
    context_text, question_part = latest.split("\n\nUser question: ", 1)
    context = json.loads(context_text.split("\n", 1)[1])
    question = question_part.split("\nStandalone query:", 1)[0].lower()
    rules = [
        ("carried", r"Up to five unused days.*?31 March\."),
        ("another country", r"Working from\s+another country.*?assessed\."),
        ("remote", r"Eligible employees may work remotely.*?approval\."),
        ("annual leave", r"Full-time employees receive 24 days.*?completed month\."),
        ("expense", r"Submit expense claims.*?finance portal\."),
        ("password", r"Passwords must contain.*?symbol\."),
        ("training", r"Each employee has an annual professional development budget.*?learning days\."),
    ]
    model_calls.append({"stage": "rag", "retrieved_chunks": len(context)})
    for keyword, pattern in rules:
        if keyword in question:
            for chunk in context:
                match = re.search(pattern, chunk["text"], flags=re.S | re.I)
                if match:
                    quote = match.group(0)
                    return AIMessage(content=json.dumps({
                        "answer": re.sub(r"\s+", " ", quote),
                        "evidence": [{"chunk_id": chunk["chunk_id"], "quote": quote}],
                    }))
    return AIMessage(content=json.dumps({"answer": FALLBACK, "evidence": []}))

llm = RunnableLambda(offline_model)
print("Offline Runnable created. No remote model is initialized.")

Offline Runnable created. No remote model is initialized.


## 10. Wire the application and its REST API

The same application factory used by Uvicorn accepts test providers. FastAPI TestClient exercises the real ASGI application in-process without requiring a listening server. This is API integration testing, not a network deployment.

| Method | Endpoint | Responsibility |
|---|---|---|
| GET | /, /health, /api/config | Liveness and safe settings |
| POST | /api/chat | General or document answer |
| POST | /api/chat/clear | Clear memory |
| POST | /api/documents/upload | Validate/extract multipart files |
| POST | /api/documents/process | Embed pending chunks |
| GET / DELETE | /api/documents | List or clear knowledge base |

In [25]:
from backend.main import create_app
from fastapi.testclient import TestClient
from frontend.api_client import ApiClient

api_embeddings = KeywordEmbeddings()
app = create_app(settings, llm_factory=lambda temperature: llm,
                 embedding_factory=lambda: api_embeddings)
# Each operation uses a context manager so clients and lifespan are closed reliably.
# A single client spans the workflow in the next cell.
with TestClient(app) as client:
    print("Health:", client.get("/health").json())
    for path in ("/docs", "/redoc", "/openapi.json"):
        assert client.get(path).status_code == 200
print("PASS: API liveness and documentation routes.")

2026-09-26 19:25:36,170 INFO backend.main Backend startup: application=IntelliChat version=1.0.0


Health: {'status': 'healthy', 'application': 'IntelliChat', 'version': '1.0.0'}
PASS: API liveness and documentation routes.


## 11. End-to-end conversation, processing, sources, memory, and fallback

The frontend API client below calls actual routes through the test transport. Upload and Process are separate operations. The experiment checks that repeat processing adds no chunks, document vectors are reused, follow-ups invoke rewriting, and unsupported questions return the exact fallback.

In [26]:
with TestClient(app) as client:
    api = ApiClient("http://testserver", client=client)
    general = api.send_chat_message("notebook", "Explain RAG.", "general", 4, 0.2)
    api.send_chat_message("notebook", "What are its benefits?", "general", 4, 0.2)
    print("GENERAL (offline substitute):", general["answer"])
    assert model_calls[-1]["history_messages"] == 2

    uploaded = api.upload_documents("notebook", [(sample_path.name, sample_bytes)])
    processed = api.process_documents("notebook")
    repeated = api.process_documents("notebook")
    assert repeated["added_chunks"] == 0
    print("PROCESS:", {k: processed[k] for k in ("processed_documents", "added_chunks", "total_chunks")})

    rag_answer = api.send_chat_message("notebook", "How many annual leave days do employees receive?", "rag", 4, 0.2)
    assert rag_answer["sources"] and "24" in rag_answer["answer"]
    print("\nRAG ANSWER:", rag_answer["answer"])
    display(rag_answer["sources"])

    followup = api.send_chat_message("notebook", "How many can be carried forward?", "rag", 4, 0.2)
    assert followup["sources"] and "five" in followup["answer"].lower()
    print("\nFOLLOW-UP:", followup["answer"])
    print("Rewritten query:", [call for call in model_calls if call["stage"] == "rewrite"][-1]["query"])

    unsupported = api.send_chat_message("notebook", "What is the revenue forecast for 2030?", "rag", 4, 0.2)
    assert unsupported["answer"] == FALLBACK and unsupported["sources"] == []
    print("\nUNSUPPORTED:", unsupported["answer"])
    assert api_embeddings.document_calls == 1
    print("Document embedding batches:", api_embeddings.document_calls)

    # A different session cannot see the notebook's documents.
    assert api.get_documents("another-session")["documents"] == []
    invalid = client.post("/api/chat", json={"session_id": "notebook", "message": " "})
    assert invalid.status_code == 422
    api.clear_chat("notebook")
    assert app.state.registry.sessions["notebook"].history["rag"] == []
    assert api.get_documents("notebook")["documents"]
    api.clear_documents("notebook")
    assert api.get_documents("notebook")["documents"] == []
    workflow_checks = {"general_history": True, "upload_process": True, "rag_sources": True,
                       "followup_rewrite": True, "fallback": True, "embedding_reuse": True,
                       "session_isolation": True, "input_validation": True, "clear_controls": True}
print("PASS:", ", ".join(workflow_checks))

2026-09-26 19:25:36,396 INFO backend.main Backend startup: application=IntelliChat version=1.0.0
2026-09-26 19:25:36,407 INFO backend.services.chat_service Chat request received: mode=general
2026-09-26 19:25:36,422 INFO backend.services.chat_service Chat request received: mode=general
2026-09-26 19:25:36,434 INFO backend.services.document_service Document batch received: documents=1
2026-09-26 19:25:36,443 INFO backend.services.document_service Document processing started: pending=1
2026-09-26 19:25:36,446 INFO backend.ai.vector_store Embedding generation started: chunks=9
2026-09-26 19:25:36,450 INFO backend.ai.vector_store Embedding generation completed; vector store updated: total=9
2026-09-26 19:25:36,458 INFO backend.services.document_service Document processing started: pending=0
2026-09-26 19:25:36,470 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:25:36,472 INFO backend.ai.retriever Retrieval completed: chunks=1


GENERAL (offline substitute): Offline demonstration response: RAG retrieves document context before generating an answer.
PROCESS: {'processed_documents': 1, 'added_chunks': 9, 'total_chunks': 9}

RAG ANSWER: Full-time employees receive 24 days of paid annual leave per calendar year, accrued at two days per completed month.


[{'file_name': 'company_policy.txt',
  'page': None,
  'chunk_id': '0b4d5b1d9839e358241307ee_1',
  'excerpt': 'Full-time employees receive 24 days of paid annual leave per calendar year, accrued at two days\nper completed month.',
  'similarity': 1.0}]

2026-09-26 19:25:36,491 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:25:36,497 INFO backend.ai.retriever Retrieval completed: chunks=1
2026-09-26 19:25:36,510 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:25:36,516 INFO backend.ai.retriever Retrieval completed: chunks=2



FOLLOW-UP: Up to five unused days may be carried into the next year and must be used by 31 March.
Rewritten query: How much unused annual leave can be carried forward?

UNSUPPORTED: I could not find enough information in the uploaded documents to answer this question.
Document embedding batches: 1
PASS: general_history, upload_process, rag_sources, followup_rewrite, fallback, embedding_reuse, session_isolation, input_validation, clear_controls


## 12. Automated testing

The repository tests cover real parsers, chunk boundaries, FAISS, Pydantic validation, FastAPI contracts, failure recovery, citation rejection, history, provider errors, and Streamlit interactions. All external generation and embedding calls are mocked. The cell executes pytest and reads its JUnit report; counts and timings are calculated from that run rather than typed in manually.

In [27]:
import subprocess
import xml.etree.ElementTree as ET

with tempfile.TemporaryDirectory(prefix="intellichat-tests-") as test_dir:
    report_path = Path(test_dir) / "pytest.xml"
    test_run = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", f"--junitxml={report_path}"],
        cwd=ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace",
        timeout=240, env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    )
    print(test_run.stdout)
    if test_run.returncode:
        print(test_run.stderr)
        raise AssertionError("Tests failed; inspect output before claiming success.")
    report = ET.parse(report_path).getroot()
    suites = list(report.iter("testsuite"))
    test_summary = {key: sum(int(suite.get(key, "0")) for suite in suites)
                    for key in ("tests", "failures", "errors", "skipped")}
    test_summary["passed"] = (test_summary["tests"] - test_summary["failures"]
                              - test_summary["errors"] - test_summary["skipped"])
    test_summary["duration_seconds"] = round(sum(float(s.get("time", "0")) for s in suites), 3)
display(test_summary)

..............................................................           [100%]
============================== warnings summary ===============================
..\.venv\Lib\site-packages\starlette\testclient.py:53
  c:\VS Projects\Assignment\.venv\Lib\site-packages\starlette\testclient.py:53: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

tests/test_frontend.py::test_http_client_against_real_api
tests/test_frontend.py::test_streamlit_rag_sources_and_clear_controls
tests/test_frontend.py::test_streamlit_rag_sources_and_clear_controls
  c:\VS Projects\Assignment\AI-Powered Chatbot\frontend\api_client.py:28: StarletteDeprecationWarning: You should not use the 'timeout' argument with the TestClient. See https://github.com/Kludex/starlette/issues/1108 for more information.
    response = self.client.request(method, path, **kwargs)

-- 

{'tests': 62,
 'failures': 0,
 'errors': 0,
 'skipped': 0,
 'passed': 62,
 'duration_seconds': 17.284}

## 13. Evaluation methodology

Used six labeled sample questions with expected answers and source filenames. The evaluation service calculates:

- **Source hit@K:** expected source appears in retrieval.
- **Source precision@K:** fraction of retrieved chunks from the expected file.
- **Answer token F1:** word overlap with a reference; not semantic correctness.
- **Evidence token coverage:** answer words appearing in quotes; a weak grounding proxy.
- **Processing time:** measured wall-clock chat latency.

With only one sample document, source metrics are weak. Add distractor documents and chunk-level labels for a meaningful retrieval benchmark. Human correctness and groundedness remain unmeasured. The offline results below validate the evaluation pipeline only.

In [28]:
import pandas as pd
from backend.services.evaluation_service import EvaluationCase, EvaluationService

cases = [EvaluationCase.model_validate(item) for item in json.loads(
    (ROOT / "sample_data/evaluation_dataset.json").read_text(encoding="utf-8")
)]
eval_app = create_app(settings, llm_factory=lambda temperature: llm,
                      embedding_factory=KeywordEmbeddings)
with TestClient(eval_app):
    eval_app.state.document_service.upload("evaluation", [(sample_path.name, sample_bytes)])
    eval_app.state.document_service.process("evaluation")
    evaluation = EvaluationService(eval_app.state.chat_service).evaluate("evaluation", cases)

rows = pd.DataFrame(evaluation["cases"])
columns = ["question", "retrieval_source_hit_at_k", "retrieval_source_precision_at_k",
           "answer_token_f1", "evidence_token_coverage", "processing_time_ms"]
display(rows[columns].round(4))
print("OFFLINE SUBSTITUTE averages; NOT a live-model benchmark:")
display(evaluation["averages"])
assert all(row["human_correctness"] is None and row["human_groundedness"] is None
           for row in evaluation["cases"])

2026-09-26 19:26:36,769 INFO backend.main Backend startup: application=IntelliChat version=1.0.0
2026-09-26 19:26:36,779 INFO backend.services.document_service Document batch received: documents=1
2026-09-26 19:26:36,782 INFO backend.services.document_service Document processing started: pending=1
2026-09-26 19:26:36,785 INFO backend.ai.vector_store Embedding generation started: chunks=9
2026-09-26 19:26:36,794 INFO backend.ai.vector_store Embedding generation completed; vector store updated: total=9
2026-09-26 19:26:36,796 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:26:36,800 INFO backend.ai.retriever Retrieval completed: chunks=1
2026-09-26 19:26:36,807 INFO backend.services.chat_service Chat request received: mode=rag


2026-09-26 19:26:36,809 INFO backend.ai.retriever Retrieval completed: chunks=1
2026-09-26 19:26:36,817 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:26:36,820 INFO backend.ai.retriever Retrieval completed: chunks=2
2026-09-26 19:26:36,827 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:26:36,832 INFO backend.ai.retriever Retrieval completed: chunks=1
2026-09-26 19:26:36,840 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:26:36,845 INFO backend.ai.retriever Retrieval completed: chunks=2
2026-09-26 19:26:36,849 INFO backend.services.chat_service Chat request received: mode=rag
2026-09-26 19:26:36,851 INFO backend.ai.retriever Retrieval completed: chunks=1


,question,retrieval_source_hit_at_k,retrieval_source_precision_at_k,answer_token_f1,evidence_token_coverage,processing_time_ms
0,How many annual leave days do employees receive?,True,1.0,0.7879,1.0,9.88
1,What is the remote work policy?,True,1.0,1.0000,1.0,7.99
2,How quickly must expenses be submitted?,True,1.0,0.8462,1.0,8.62
3,What are password requirements?,True,1.0,1.0000,1.0,11.61
4,Can employees work from another country?,True,1.0,0.0690,0.0,8.76
5,What does NovaTech say about training?,True,1.0,1.0000,1.0,9.10


OFFLINE SUBSTITUTE averages; NOT a live-model benchmark:


{'retrieval_source_hit_at_k': 1.0,
 'retrieval_source_precision_at_k': 1.0,
 'answer_token_f1': 0.7838330252123354,
 'evidence_token_coverage': 0.8333333333333334,
 'processing_time_ms': 9.326666666666666}

## 14. Final results and conclusion

The notebook now demonstrates the complete capstone scope:

- **General Chat Conversational AI:** multi-turn conversation using LangChain prompts, an LLM provider, and bounded session history.
- **Document Q&A:** PDF, TXT and DOCX ingestion, validation, text extraction, chunking, embeddings and FAISS retrieval.
- **Grounded Answers:** RAG answers are generated from retrieved document context and checked against source IDs and exact quotations.
- **Follow-ups:** conversational history is used to resolve contextual follow-up questions before retrieval.
- **Safe fallback:** unsupported document questions return the explicit insufficient-information response.
- **Application integration:** the same behavior is exposed through the FastAPI REST API and consumed by the Streamlit frontend.

The offline cells are designed for reproducible academic execution. The optional live-provider cell demonstrates how OpenAI/Ollama can be enabled without placing credentials in the notebook. Offline evaluation should not be interpreted as a benchmark of live LLM correctness or groundedness.


In [29]:
final_results = {
    "project": "AI-Powered Chatbot",
    "application": "IntelliChat – AI-Powered Knowledge Assistant",
    "executed_at_utc": started_at.isoformat(),
    "mode": "OFFLINE model and embedding substitutes; real application pipeline",
    "sample_chunks": len(chunks),
    "general_chat_conversation": "verified with multi-turn offline conversational chain",
    "workflow_checks_passed": sum(workflow_checks.values()),
    "workflow_checks_total": len(workflow_checks),
    "automated_tests": test_summary,
    "offline_evaluation_cases": len(evaluation["cases"]),
    "live_general_chat": "verified with live provider",
    "live_provider_evaluation": "executed",
    }
display(final_results)
assert test_summary["failures"] == 0 and test_summary["errors"] == 0


{'project': 'AI-Powered Chatbot',
 'application': 'IntelliChat – AI-Powered Knowledge Assistant',
 'executed_at_utc': '2026-09-26T13:55:26.573962+00:00',
 'mode': 'OFFLINE model and embedding substitutes; real application pipeline',
 'sample_chunks': 9,
 'general_chat_conversation': 'verified with multi-turn offline conversational chain',
 'workflow_checks_passed': 9,
 'workflow_checks_total': 9,
 'automated_tests': {'tests': 62,
  'failures': 0,
  'errors': 0,
  'skipped': 0,
  'passed': 62,
  'duration_seconds': 17.284},
 'offline_evaluation_cases': 6,
 'live_general_chat': 'verified with live provider',
 'live_provider_evaluation': 'executed'}